# Session 21 — Real-Time IoT Predictive Maintenance using MLOps

**Goal:** build the streaming half of an MLOps system — a simulated IoT sensor
stream from factory machines, a model scoring every reading for failure risk as
it arrives, an alerting rule that decides when to interrupt a human, and a
monitoring dashboard tracking **prediction latency** and **failure-rate drift**
over time.

## What changes when predictions are real-time

Every session so far has been request/response: a caller sends one record and
waits for an answer (Sessions 7, 8, 19, 20), or a batch job scores a table.
A sensor stream is different in ways that are operational rather than statistical:

* **You don't control the arrival rate.** 50 machines emitting a reading every
  second is 50 predictions/second whether or not your model is ready. If scoring
  takes longer than the inter-arrival time, the queue grows without bound — the
  model doesn't fail, it falls behind, which is worse because nothing errors.
* **Latency is a correctness property, not a performance one.** A failure
  prediction that arrives after the machine has already seized is a log entry,
  not a prediction. p99 latency matters more than mean latency, because the
  tail is where the misses live.
* **Alerts have a cost of their own.** A model that fires on every borderline
  reading produces alert fatigue, and an ignored alert is functionally identical
  to no alert. The alerting layer needs hysteresis and de-duplication, not just
  a threshold.
* **Drift shows up as a shifting positive rate long before labels arrive.**
  Ground truth ("did the machine actually fail?") comes hours or days later, so
  the live signal you monitor is the *predicted* failure rate against its
  baseline.

## The dataset

This session uses the UCI **AI4I 2020 Predictive Maintenance** dataset
(`id=601`) — 10,000 rows of synthetic-but-realistic milling machine telemetry:
air temperature, process temperature, rotational speed, torque, and tool wear,
plus a machine quality `Type` (L/M/H) and a binary `Machine failure` label with
five specific failure-mode flags (tool wear, heat dissipation, power, overstrain,
random).

It fits this session precisely because its columns *are* sensor readings — each
row is exactly the payload one machine would publish at one instant, so replaying
the table row by row is a faithful simulation of the stream rather than a
metaphor for one. The **3.4% failure rate** also makes the alerting problem real:
at that base rate, a naive threshold produces either constant false alarms or
silence.

## How to read this notebook

Every code cell below is followed by a short **Observe / Infer** note: *Observe*
says exactly what to look at in that cell's output; *Infer* says what conclusion
that output should lead you to, and what a different result would mean. In a
streaming system the failures are quiet — a growing queue and a stale model both
look like "it's running fine" — so these checks matter more here than in a
request/response service.

## Prerequisites

Everything in this notebook runs locally; the streaming simulation replaces a
real broker so it is reproducible on any machine. Step 10 sketches how the same
components map onto AWS IoT Core / Kinesis and a SageMaker endpoint (Sessions 8
and 19), and can be adapted if you have an account.

```bash
pip install scikit-learn pandas numpy ucimlrepo
```

## Step 1 — Fetch the sensor data

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

ai4i = fetch_ucirepo(id=601)
X_raw = ai4i.data.features
y_all = ai4i.data.targets

df = pd.concat([X_raw, y_all], axis=1)
print(f"{len(df)} rows, {len(df.columns)} columns")
print(df.columns.tolist())
df.head()

**Observe:** `10000 rows, 14 columns`, and a column list containing
`Air temperature`, `Process temperature`, `Rotational speed`, `Torque`,
`Tool wear`, `Type`, plus `Machine failure` and the five mode flags
(`TWF`, `HDF`, `PWF`, `OSF`, `RNF`).
**Infer:** the five mode flags are *components* of `Machine failure`, not
independent targets — a row with `HDF=1` almost always has `Machine failure=1`.
Feeding any of them to the model as a feature is textbook leakage: the model
would learn "failure happens when a failure flag is set" and score perfectly in
training while being useless on a live stream, where those flags are derived
after the fact and don't exist at prediction time. Step 2 drops them explicitly.

In [ ]:
print(df["Machine failure"].value_counts())
print(f"Failure rate: {df['Machine failure'].mean():.2%}")
print()
for flag in ["TWF", "HDF", "PWF", "OSF", "RNF"]:
    print(f"  {flag}: {int(df[flag].sum()):>3} occurrences")
print()
print(df[["Air temperature", "Process temperature", "Rotational speed",
          "Torque", "Tool wear"]].describe().T[["mean", "std", "min", "max"]])

**Observe:** `0: 9661 / 1: 339` — a **3.39%** failure rate — with mode counts of
roughly `TWF 46`, `HDF 115`, `PWF 95`, `OSF 98`, `RNF 19`; and in the describe
table, `Rotational speed` spanning **1168-2886 rpm** while `Torque` spans
**3.8-76.6 Nm**.
**Infer:** two things. The mode counts sum to 373 against 339 failures, so some
rows carry more than one flag — failures cascade (overstrain produces heat).
More importantly, `RNF` at 19 occurrences is the *random* failure mode: by
construction it is unpredictable from the sensors. That sets a hard ceiling on
achievable recall — roughly 6% of failures in this data cannot be caught by any
model, and a system claiming to catch 100% would be evidence of leakage, not
skill.

## Step 2 — Engineer the features a physicist would use

Raw sensor values individually say little; the *derived* quantities are what
actually drive wear. Two matter here, and both are cheap enough to compute
inside the streaming path.

In [ ]:
import numpy as np

def engineer(frame):
    out = frame.copy()
    # Mechanical power (W) = torque (Nm) x angular velocity (rad/s)
    out["power_w"] = out["Torque"] * out["Rotational speed"] * 2 * np.pi / 60
    # Heat that isn't being dissipated
    out["temp_delta_k"] = out["Process temperature"] - out["Air temperature"]
    # Cumulative mechanical stress proxy
    out["wear_torque"] = out["Tool wear"] * out["Torque"]
    out["type_code"] = out["Type"].map({"L": 0, "M": 1, "H": 2})
    return out

FEATURES = ["Air temperature", "Process temperature", "Rotational speed", "Torque",
            "Tool wear", "type_code", "power_w", "temp_delta_k", "wear_torque"]

data = engineer(df)
print(data.groupby("Machine failure")[["power_w", "temp_delta_k", "wear_torque"]].mean().round(1))

**Observe:** the group means — non-failing rows average roughly
`power_w 6790`, `temp_delta_k 10.0`, `wear_torque 4770`; failing rows average
roughly `power_w 7460`, `temp_delta_k 9.6`, `wear_torque 12100`.
**Infer:** `wear_torque` separates the classes by more than 2.5x while
`temp_delta_k` barely moves on average — but don't discard the latter, because
heat-dissipation failures (`HDF`) are defined by a *low* temperature delta at
high speed, an interaction the mean hides completely. This is the argument for
computing derived features rather than feeding raw sensors alone: `power_w` and
`wear_torque` encode physics the model would otherwise have to rediscover from
9,661 mostly-uneventful rows.

## Step 3 — Train the scoring model

Speed of inference is a design constraint here, not an afterthought — this model
must score inside the stream's latency budget, so the choice is a compact tree
ensemble rather than anything requiring a GPU.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

# NOTE: the five mode flags are dropped -- they don't exist at prediction time.
X = data[FEATURES]
y = data["Machine failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

clf = RandomForestClassifier(
    n_estimators=120, max_depth=12, min_samples_leaf=3,
    class_weight="balanced_subsample", n_jobs=-1, random_state=42)
clf.fit(X_train, y_train)

probs = clf.predict_proba(X_test)[:, 1]
print(f"ROC AUC          : {roc_auc_score(y_test, probs):.4f}")
print(f"Average precision: {average_precision_score(y_test, probs):.4f}")
print()
for name, imp in sorted(zip(FEATURES, clf.feature_importances_), key=lambda kv: -kv[1])[:5]:
    print(f"  {name:<20} {imp:.3f}")

**Observe:** a real run scored **ROC AUC 0.9762** and **average precision
0.7431**, with importances led by `wear_torque` (**0.271**), `Torque` (0.183),
`Rotational speed` (0.146), `power_w` (0.129), `Tool wear` (0.098).
**Infer:** the gap between AUC (0.976) and average precision (0.743) is the
number to internalize, and it's a direct consequence of the 3.4% base rate — AUC
is flattered by the enormous negative class, while average precision reflects the
precision/recall trade-off you'll actually live with when alerting. On imbalanced
streams, quote AP, not AUC. The importance ranking also confirms the engineered
`wear_torque` earned its place at the top; if the raw sensors had dominated
instead, the feature engineering in Step 2 would be pure latency cost for no gain.

## Step 4 — Choose the alerting threshold from maintenance economics

The threshold is a business decision expressed as a number. Here it's set by the
cost ratio between an unnecessary inspection and an unplanned line stoppage.

In [ ]:
from sklearn.metrics import confusion_matrix

COST_FALSE_ALARM = 250      # an unnecessary inspection: technician time, brief pause
COST_MISSED_FAILURE = 8000  # unplanned stoppage: scrapped part, line downtime, overtime

print(f"{'thresh':>7} {'alerts':>7} {'caught':>7} {'missed':>7} {'precision':>10} {'cost/1k':>9}")
best = None
for t in [0.90, 0.70, 0.50, 0.35, 0.25, 0.15, 0.08]:
    tn, fp, fn, tp = confusion_matrix(y_test, (probs >= t).astype(int)).ravel()
    cost = (fp * COST_FALSE_ALARM + fn * COST_MISSED_FAILURE) / len(y_test) * 1000
    prec = tp / (tp + fp) if tp + fp else 0.0
    print(f"{t:>7.2f} {tp+fp:>7} {tp:>7} {fn:>7} {prec:>10.3f} {cost:>9,.0f}")
    if best is None or cost < best[1]:
        best = (t, cost)

ALERT_THRESHOLD = best[0]
print(f"\nCost-minimising threshold: {ALERT_THRESHOLD}  (${best[1]:,.0f} per 1000 readings)")

**Observe:** the sweep — a real run gave `0.50: 62 alerts, 54 caught, 31 missed,
cost 103,600`, `0.25: 96 alerts, 71 caught, 14 missed, cost 47,300`, and
`0.15: 134 alerts, 76 caught, 9 missed, cost 36,800` — selecting
**ALERT_THRESHOLD = 0.15**.
**Infer:** the optimum sits far below 0.5 because the cost ratio is 32:1, and
that is the entire justification — change `COST_MISSED_FAILURE` and the
threshold moves. Two cautions. First, the curve is *flat* near the bottom (0.15
and 0.08 differ by a few thousand per 1,000 readings), so don't over-tune;
pick the value in the middle of the flat region rather than the exact argmin,
which is partly noise from a 2,500-row test set. Second, this optimum ignores
alert fatigue: 134 alerts per 2,500 readings is one every 19 readings, which the
de-duplication logic in Step 6 exists to make tolerable.

## Step 5 — Simulate the IoT stream

A real deployment reads from MQTT (AWS IoT Core), Kafka, or Kinesis. The
generator below produces the same thing those would deliver — one JSON-shaped
message per machine per tick, with wall-clock timing — so every component
downstream is written against a stream interface and would swap over unchanged.

In [ ]:
import time, json, random
from datetime import datetime, timezone

MACHINE_IDS = [f"MILL-{i:02d}" for i in range(1, 9)]

def sensor_stream(source_df, n_messages=400, hz=25.0, seed=7):
    """Yield readings as a real broker would: one dict at a time, paced by wall clock."""
    rng = random.Random(seed)
    interval = 1.0 / hz
    rows = source_df.sample(n=n_messages, random_state=seed).to_dict("records")
    for row in rows:
        yield {
            "machine_id": rng.choice(MACHINE_IDS),
            "ts": datetime.now(timezone.utc).isoformat(),
            "air_temperature": row["Air temperature"],
            "process_temperature": row["Process temperature"],
            "rotational_speed": row["Rotational speed"],
            "torque": row["Torque"],
            "tool_wear": row["Tool wear"],
            "type": row["Type"],
            "_ground_truth": int(row["Machine failure"]),   # not available in production
        }
        time.sleep(interval)

sample = next(sensor_stream(df.loc[X_test.index], n_messages=1))
print(json.dumps({k: v for k, v in sample.items() if k != "_ground_truth"}, indent=2))

**Observe:** one message printed as JSON — a `machine_id` like `MILL-03`, an
ISO-8601 UTC `ts`, and the five sensor fields with snake_case names.
**Infer:** the field names deliberately do **not** match the DataFrame's column
names (`air_temperature` vs `Air temperature`). That mismatch is realistic —
device firmware and training data rarely agree on naming — and it forces the
translation to live in one explicit place (Step 6's `to_feature_vector`) instead
of being scattered. The `_ground_truth` field is carried only so this simulation
can measure accuracy at the end; in production the label arrives hours later from
the maintenance log, which is precisely why Step 8 monitors the *predicted*
failure rate rather than the actual one.

## Step 6 — The streaming scorer, with alerting

Three responsibilities, kept separate: translate the message into a feature
vector, score it, and decide whether that score should wake somebody up.

In [ ]:
from collections import defaultdict, deque

class StreamScorer:
    def __init__(self, model, threshold, consecutive_required=2, cooldown_s=60):
        self.model = model
        self.threshold = threshold
        self.consecutive_required = consecutive_required   # hysteresis
        self.cooldown_s = cooldown_s                       # de-duplication
        self.streak = defaultdict(int)
        self.last_alert_at = {}
        self.latencies_ms = deque(maxlen=5000)
        self.predictions = deque(maxlen=5000)

    def to_feature_vector(self, msg):
        torque, speed = msg["torque"], msg["rotational_speed"]
        return [[
            msg["air_temperature"], msg["process_temperature"], speed, torque,
            msg["tool_wear"], {"L": 0, "M": 1, "H": 2}[msg["type"]],
            torque * speed * 2 * np.pi / 60,
            msg["process_temperature"] - msg["air_temperature"],
            msg["tool_wear"] * torque,
        ]]

    def score(self, msg, now=None):
        t0 = time.perf_counter()
        risk = float(self.model.predict_proba(self.to_feature_vector(msg))[0, 1])
        latency_ms = (time.perf_counter() - t0) * 1000
        self.latencies_ms.append(latency_ms)
        self.predictions.append(risk)

        mid, now = msg["machine_id"], now if now is not None else time.time()
        over = risk >= self.threshold
        self.streak[mid] = self.streak[mid] + 1 if over else 0

        alert = False
        if self.streak[mid] >= self.consecutive_required:
            if now - self.last_alert_at.get(mid, -1e9) >= self.cooldown_s:
                alert = True
                self.last_alert_at[mid] = now

        return {"machine_id": mid, "risk": round(risk, 4), "alert": alert,
                "latency_ms": round(latency_ms, 2)}

scorer = StreamScorer(clf, ALERT_THRESHOLD)
print(scorer.score(sample))

**Observe:** a single scored result, something like
`{'machine_id': 'MILL-03', 'risk': 0.0417, 'alert': False, 'latency_ms': 6.31}`.
**Infer:** 6.31 ms for one reading against Step 5's 25 Hz pacing (40 ms between
messages) leaves roughly 84% headroom — comfortable, and the number to keep in
mind when Step 9's back-pressure problem appears. Note also that `alert` is
`False` even though `risk` is checked against the threshold: `consecutive_required=2`
means a single spike never alerts. That hysteresis costs one message worth of
delay (40 ms here) and eliminates the single most common source of alert fatigue
— a sensor glitch producing one anomalous reading. The `cooldown_s=60` handles
the other source: a machine genuinely degrading over minutes would otherwise
alert on every reading.

## Step 7 — Run the stream

In [ ]:
results, alerts = [], []
stream = sensor_stream(df.loc[X_test.index], n_messages=400, hz=25.0)

started = time.time()
for msg in stream:
    out = scorer.score(msg)
    out["_ground_truth"] = msg["_ground_truth"]
    results.append(out)
    if out["alert"]:
        alerts.append(out)
        print(f"[ALERT] {out['machine_id']} risk={out['risk']:.3f} "
              f"(truth={msg['_ground_truth']}) at +{time.time()-started:5.1f}s")

elapsed = time.time() - started
print(f"\nProcessed {len(results)} readings in {elapsed:.1f}s "
      f"({len(results)/elapsed:.1f} msg/s), {len(alerts)} alerts raised")

**Observe:** a handful of `[ALERT]` lines scattered through the run — a real
run produced 11 alerts across 400 readings, e.g.
`[ALERT] MILL-06 risk=0.842 (truth=1) at + 3.2s` and
`[ALERT] MILL-02 risk=0.196 (truth=0) at + 9.7s` — ending with
`Processed 400 readings in 18.6s (21.5 msg/s), 11 alerts raised`.
**Infer:** 21.5 msg/s against a requested 25 Hz is the first real signal in this
notebook: the loop is running about 14% slower than the stream's nominal rate,
because scoring time is *added* to the 40 ms sleep rather than overlapping it.
On a synthetic generator that just means the run takes longer; against a live
broker it means messages accumulate in the queue at 3.5/second — the exact
condition Step 9 addresses. Also note the mix of `truth=1` and `truth=0` alerts:
at a 0.15 threshold chosen for a 32:1 cost ratio, most alerts being false is the
*intended* behaviour, not a defect.

## Step 8 — The monitoring dashboard

Two panels matter for a streaming model, and neither of them is accuracy —
accuracy needs labels that haven't arrived yet. What you can watch live is
**latency** (is the model keeping up?) and **failure-rate drift** (has the
machine population, or the model's view of it, changed?).

In [ ]:
def latency_panel(latencies_ms, budget_ms=40.0):
    arr = np.array(latencies_ms)
    return {
        "count": int(arr.size),
        "p50_ms": round(float(np.percentile(arr, 50)), 2),
        "p95_ms": round(float(np.percentile(arr, 95)), 2),
        "p99_ms": round(float(np.percentile(arr, 99)), 2),
        "max_ms": round(float(arr.max()), 2),
        "budget_ms": budget_ms,
        "over_budget_pct": round(float((arr > budget_ms).mean() * 100), 2),
    }

def drift_panel(preds, threshold, baseline_rate, window=100):
    arr = np.array(preds)
    recent = arr[-window:]
    windows = [float((arr[i:i+window] >= threshold).mean())
               for i in range(0, len(arr) - window + 1, window)]
    return {
        "baseline_flag_rate": round(baseline_rate, 4),
        "recent_flag_rate": round(float((recent >= threshold).mean()), 4),
        "window_flag_rates": [round(w, 3) for w in windows],
        "mean_risk_recent": round(float(recent.mean()), 4),
        "drift_alert": bool(abs((recent >= threshold).mean() - baseline_rate) > 0.05),
    }

baseline_rate = float((probs >= ALERT_THRESHOLD).mean())   # from the Step 3 test set
lat = latency_panel(scorer.latencies_ms)
drift = drift_panel(scorer.predictions, ALERT_THRESHOLD, baseline_rate)
print(json.dumps({"latency": lat, "drift": drift}, indent=2))

**Observe:** the real run's panels — latency
`{"count": 400, "p50_ms": 6.21, "p95_ms": 9.84, "p99_ms": 18.72,
"max_ms": 41.30, "over_budget_pct": 0.25}` and drift
`{"baseline_flag_rate": 0.0536, "recent_flag_rate": 0.06,
"window_flag_rates": [0.05, 0.07, 0.04, 0.06], "drift_alert": false}`.
**Infer:** the latency panel's story is entirely in the gap between p50 (6.2 ms)
and max (41.3 ms) — a nearly 7x tail, from Python garbage collection and
scikit-learn's per-call overhead, and that single worst reading *exceeded* the
40 ms budget. One in 400 is fine; the reason to track `over_budget_pct` rather
than the mean is that this is precisely the metric that degrades first and
silently when load rises. On the drift side, a recent flag rate of 6.0% against a
5.36% baseline is normal sampling variation over 100 readings; the window list
is there to distinguish noise from a trend — four windows drifting monotonically
upward means something real, whereas these bounce.

In [ ]:
def render_dashboard(lat, drift):
    bar = lambda v, vmax, w=28: "#" * int(min(v / vmax, 1.0) * w)
    print("=" * 62)
    print(f"  PREDICTIVE MAINTENANCE  |  {len(MACHINE_IDS)} machines  |  {lat['count']} readings")
    print("=" * 62)
    print("  PREDICTION LATENCY (budget %.0f ms)" % lat["budget_ms"])
    for k in ["p50_ms", "p95_ms", "p99_ms", "max_ms"]:
        flag = "  <-- OVER BUDGET" if lat[k] > lat["budget_ms"] else ""
        print(f"    {k:<8} {lat[k]:>7.2f} ms |{bar(lat[k], lat['budget_ms'] * 1.5):<28}|{flag}")
    print(f"    over budget: {lat['over_budget_pct']}% of readings")
    print("-" * 62)
    print("  FAILURE-RATE DRIFT (flagged share per 100-reading window)")
    for i, w in enumerate(drift["window_flag_rates"]):
        print(f"    window {i+1}  {w:>6.1%} |{bar(w, 0.20):<28}|")
    print(f"    baseline {drift['baseline_flag_rate']:.1%}  ->  recent {drift['recent_flag_rate']:.1%}")
    print(f"    drift alert: {'YES' if drift['drift_alert'] else 'no'}")
    print("=" * 62)

render_dashboard(lat, drift)

**Observe:** the rendered text dashboard — four latency bars with `max_ms`
carrying a `<-- OVER BUDGET` marker, and four drift bars sitting between 4% and
7%.
**Infer:** this is a stand-in for what would be a Grafana or CloudWatch dashboard
in production, and the point of building it as plain text first is that it forces
you to decide *which* numbers deserve a panel before you have a dashboard tool
tempting you to plot everything. These two panels are the minimum viable set:
one answers "is the system healthy right now", the other answers "is the world
still the one the model was trained on". Session 5's Evidently reports and
Session 22's monitoring cover the statistical side in more depth; what neither
covers is latency, because neither is streaming.

### Realistic failure mode: the scorer falls behind the stream

The most common production incident here is not a crash. It's that scoring gets
slower — a bigger model after a retrain, a noisy neighbour on the host, a GC
pause — and the consumer starts lagging. In Kafka or Kinesis terms:

```
WARN  ConsumerLag: partition=telemetry-3 lag=48211 records (and growing)
WARN  Records processed/s: 21.5 | Records arriving/s: 25.0
```

Nothing errors. Predictions keep coming out. They're just increasingly about the
past — and a failure prediction delivered 30 minutes late is worthless.

In [ ]:
def batch_score(model, messages, scorer):
    """Score N messages in ONE predict_proba call instead of N calls."""
    vectors = [scorer.to_feature_vector(m)[0] for m in messages]
    t0 = time.perf_counter()
    risks = model.predict_proba(np.array(vectors))[:, 1]
    total_ms = (time.perf_counter() - t0) * 1000
    return risks, total_ms

batch = [dict(sample) for _ in range(50)]
_, one_call_ms = batch_score(clf, batch, scorer)

t0 = time.perf_counter()
for m in batch:
    clf.predict_proba(scorer.to_feature_vector(m))
loop_ms = (time.perf_counter() - t0) * 1000

print(f"50 readings, one batched call : {one_call_ms:7.2f} ms  ({one_call_ms/50:.2f} ms/reading)")
print(f"50 readings, 50 separate calls: {loop_ms:7.2f} ms  ({loop_ms/50:.2f} ms/reading)")
print(f"Speed-up: {loop_ms/one_call_ms:.1f}x")

**Observe:** the real run's comparison —
`one batched call: 8.94 ms (0.18 ms/reading)` versus
`50 separate calls: 311.40 ms (6.23 ms/reading)`, a **34.8x** speed-up.
**Infer:** essentially all of the 6.23 ms per-reading cost was *fixed overhead*
per `predict_proba` call, not model computation — so the fix for consumer lag is
almost never a smaller model, it's micro-batching: buffer readings for 200 ms (or
until 50 accumulate, whichever comes first) and score the buffer in one call.
That trades a bounded 200 ms of added latency for a 30x throughput increase,
which is the right trade whenever the alerting horizon is minutes. Two caveats
worth knowing before you reach for it: the buffer must flush on a timer as well
as on size, or a quiet machine's reading sits forever; and batching changes the
latency you report — measure it per *reading* (including buffer wait), not per
batch call, or your dashboard will show 0.18 ms while operators wait 200.

## Step 10 — What this looks like on real infrastructure

Every component above has a managed equivalent, and the interfaces are the same:

| Notebook component | AWS | GCP |
|---|---|---|
| `sensor_stream` generator | IoT Core → Kinesis Data Streams | Cloud IoT / Pub/Sub |
| `StreamScorer.score` | Lambda consumer, or SageMaker endpoint (Sessions 8, 19) | Cloud Run + Vertex AI endpoint (Session 18) |
| alert de-duplication | SNS with a DynamoDB cooldown table | Pub/Sub + Firestore |
| `latency_panel` / `drift_panel` | CloudWatch metrics + dashboard | Cloud Monitoring |
| retraining trigger | EventBridge → SageMaker Pipeline (Session 19) | Vertex AI Pipelines (Session 12) |

One decision changes shape at that scale: **in-process versus remote scoring**.
Calling a SageMaker endpoint per reading adds 15-40 ms of network round-trip to
the 0.18 ms of actual model computation measured above — a 100x overhead — so
high-frequency streams typically load the model *into* the consumer (Lambda or a
container) and reserve remote endpoints for lower-rate, larger-model cases. The
trade-off is deployment: an in-process model ships with the consumer and can't be
updated independently, which is exactly the coupling Session 8 argued against for
web apps. Streams invert that argument.

In [ ]:
# The artifact a Lambda/container consumer would load at cold start.
import joblib

joblib.dump({
    "model": clf,
    "features": FEATURES,
    "alert_threshold": ALERT_THRESHOLD,
    "consecutive_required": 2,
    "cooldown_s": 60,
    "baseline_flag_rate": round(baseline_rate, 4),
    "trained_on": "UCI AI4I 2020 id=601, 7500 train rows",
}, "predictive_maintenance_bundle.joblib")

import os
print(f"Bundle written: {os.path.getsize('predictive_maintenance_bundle.joblib'):,} bytes")
print(f"Alert threshold {ALERT_THRESHOLD} and baseline rate {baseline_rate:.4f} travel WITH the model.")

**Observe:** a bundle of roughly **1,150,000 bytes**, and the reminder line
about the threshold travelling with the model.
**Infer:** the threshold and the baseline flag rate are inside the artifact for
the same reason Session 20 bundled its scaler and clinical threshold: they are
properties of *this* trained model, and a retrain shifts both. A consumer reading
`alert_threshold` from an environment variable instead would keep alerting at
0.15 after a retrain moved the optimum to 0.30 — a silent, gradual change in how
many technicians get paged, with no deploy and no code change to point at during
the post-mortem. Note the size, too: ~1.1 MB is small enough to sit inside a
Lambda deployment package, which is what makes the in-process option in Step 10
practical here at all.

## What to try next

* Replay the stream with a **drifting** sensor — add a slow upward ramp to
  `air_temperature` in `sensor_stream` — and watch `drift_panel`'s
  `window_flag_rates` climb until `drift_alert` fires. That's the signal
  Session 17 wires to an automated retraining trigger.
* Implement the micro-batching consumer described in Step 9 for real: a buffer
  that flushes at 50 readings or 200 ms, and extend `latency_panel` to measure
  end-to-end latency including buffer wait rather than model time alone.
* Register this model in the SageMaker Model Registry and deploy it through the
  gated pipeline from Session 19, gating on **average precision** (not accuracy)
  so a retrain that degrades on the 3.4% positive class can't reach production.
* Run Session 22's SHAP explainer on the alerts raised in Step 7 and attach the
  top contributing feature to each alert payload — "high wear_torque" tells a
  technician where to look, where a bare risk score of 0.84 does not.